In [ ]:
%load_ext autoreload
%autoreload 2

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import torch
from pathlib import Path
from scipy.optimize import curve_fit
from sklearn.model_selection import train_test_split
from torch_geometric.loader import DataLoader

from analysis.gravnet.model import NeutrinoGravNetRegressionFASER, NeutrinoGravNetNodesFaser
from analysis.utils.utils import get_torch_path, get_weights_path, get_figures_path

WEIGHTS_DIR = "gravnet_regression_faser_huber1.0_nofaser_final"
RUN = 20000
DATA_TYPE = "all"
# Shards used for Stage-2 regressor training — must match the training script.
CHUNKS = list(range(0, 60))
BATCH_SIZE = 8

# Required only for _binaryprob runs — set to the binary classifier best_model.pt path.
# Leave as None for all other run types.
BINARY_CLASSIFIER_WEIGHTS = None   # e.g. get_weights_path() / "gravnet_binary_classifier_faser/best_model.pt"
TRUTH3B_CLASSIFIER_WEIGHTS = None   # e.g. get_weights_path() / "gravnet_truth3b_classifier_faser/best_model.pt"

TARGET_NAMES = ["E_nu", "E_lepton", "E_roe"]
TARGET_LATEX = [r"$E_\nu$", r"$E_\mathrm{lep}$", r"$E_\mathrm{roe}$"]
TARGET_UNITS = "TeV"

ENERGY_BINS_TEV = [
    (0.01, 0.05), (0.05, 0.1), (0.1, 0.2),
    (0.2,  0.3),  (0.3,  0.5), (0.5,  0.7),
    (0.7,  1.0),  (1.0,  1.5), (1.5,  3.0),
]

SAVE_EVAL_HISTORY = True   # set True to append stats to eval_history.jsonl

device = torch.device("cuda:0" if torch.cuda.is_available() else "cpu")
print(f"Device: {device}")

weights_path = get_weights_path() / WEIGHTS_DIR
torch_path   = get_torch_path()
figures_path = get_figures_path() / WEIGHTS_DIR
figures_path.mkdir(parents=True, exist_ok=True)

print(f"Weights : {weights_path}")
print(f"Figures : {figures_path}")

C1 = "#353D4C"   # E_nu
C2 = "#E17883"   # E_lepton
C3 = "#5691D9"   # E_roe

**Training curves**

In [ ]:
import seaborn as sns

sns.set_style("ticks")
sns.set_context("paper", font_scale=1.2)
plt.rcParams.update({
    'font.family': 'serif',
    'font.serif': ['DejaVu Serif', 'Times New Roman', 'Times'],
    'mathtext.fontset': 'dejavuserif',
    'axes.linewidth': 0.8,
    'xtick.major.width': 0.8,
    'ytick.major.width': 0.8,
    'xtick.direction': 'in',
    'ytick.direction': 'in',
    'figure.dpi': 350,
})

C1 = "#353D4C"   # train / E_nu
C2 = "#E17883"   # val   / E_lepton
C3 = "#5691D9"   # E_roe
MARKER_KW = dict(markersize=4, markerfacecolor='white', markeredgewidth=1.2)

metrics = np.load(weights_path / "training_metrics.npz")
epochs  = np.arange(1, len(metrics["train_loss"]) + 1)

_ckpt_meta = torch.load(weights_path / "best_model.pt", weights_only=False)
_loss_fn = _ckpt_meta.get("loss_fn")
loss_label = f"Loss ({_loss_fn})" if _loss_fn is not None else "Loss"

fig, axes = plt.subplots(2, 2, figsize=(12, 8))

ax = axes[0, 0]
ax.plot(epochs, metrics["train_loss"], marker='o', linewidth=0.8, color=C1, label="Train total", **MARKER_KW)
ax.plot(epochs, metrics["val_loss"],   marker='s', linewidth=0.8, color=C2, label="Val total",   **MARKER_KW)
if "train_loss_t1" in metrics:
    ax.plot(epochs, metrics["train_loss_t1"], linestyle='--', linewidth=0.8, color=C1, alpha=0.6, label="Train t1")
    ax.plot(epochs, metrics["train_loss_t2"], linestyle=':',  linewidth=0.8, color=C1, alpha=0.6, label="Train t2")
    ax.plot(epochs, metrics["val_loss_t1"],   linestyle='--', linewidth=0.8, color=C2, alpha=0.6, label="Val t1")
    ax.plot(epochs, metrics["val_loss_t2"],   linestyle=':',  linewidth=0.8, color=C2, alpha=0.6, label="Val t2")
_all_loss = list(metrics["train_loss"]) + list(metrics["val_loss"])
if min(_all_loss) > 0:
    ax.set_yscale("log")
ax.set_xlabel("Epoch")
ax.set_ylabel(loss_label)
ax.legend(frameon=False, fontsize=7)
ax.set_title(f"{loss_label} (reparam space)")
ax.spines[['top', 'right']].set_visible(False)
ax.grid(True, alpha=0.3)

ax = axes[0, 1]
ax.plot(epochs, metrics["train_rmse"], marker='o', linewidth=0.8, color=C1, label="Train", **MARKER_KW)
ax.plot(epochs, metrics["val_rmse"],   marker='s', linewidth=0.8, color=C2, label="Val",   **MARKER_KW)
ax.set_xlabel("Epoch")
ax.set_ylabel("RMSE")
ax.legend(frameon=False)
ax.set_title("RMSE")
ax.spines[['top', 'right']].set_visible(False)
ax.grid(True, alpha=0.3)

ax = axes[1, 0]
for i, (latex, c, m) in enumerate(zip(TARGET_LATEX, [C1, C2, C3], ['o', 's', '^'])):
    ax.plot(epochs, metrics["val_rel_err"][:, i], marker=m, linewidth=0.8, color=c, label=latex, **MARKER_KW)
ax.set_xlabel("Epoch")
ax.set_ylabel("Mean |pred - true| / true")
ax.legend(frameon=False)
ax.set_title("Val relative error (physical space)")
ax.spines[['top', 'right']].set_visible(False)
ax.grid(True, alpha=0.3)

ax = axes[1, 1]
for i, (latex, c, m) in enumerate(zip(TARGET_LATEX, [C1, C2, C3], ['o', 's', '^'])):
    ax.plot(epochs, metrics["val_resolution"][:, i], marker=m, linewidth=0.8, color=c, label=latex, **MARKER_KW)
ax.set_xlabel("Epoch")
ax.set_ylabel("std(rel. error)")
ax.legend(frameon=False)
ax.set_title("Val resolution (std, physical space)")
ax.spines[['top', 'right']].set_visible(False)
ax.grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig(figures_path / "training_curves.png", dpi=500, bbox_inches='tight')
plt.show()
print("Saved: training_curves.png")

**Model and data**

In [ ]:
import torch.nn.functional as F

# Load checkpoint
checkpoint = torch.load(weights_path / "best_model.pt", weights_only=False)
print(f"Best model from epoch {checkpoint['epoch'] + 1},  "
      f"val_loss={checkpoint['val_loss']:.4f},  "
      f"val_rmse={checkpoint['val_rmse']:.4f}")
if "loss_weights" in checkpoint:
    w_enu, w_logit = checkpoint["loss_weights"]
    print(f"Loss weights: w_enu={w_enu:.4f}, w_logit={w_logit:.4f}")
if "parametrisation" in checkpoint:
    print(f"Parametrisation: {checkpoint['parametrisation']}")

# Derive pooling from directory name so it always matches the trained run
pooling = "sum" if "_sum_" in WEIGHTS_DIR else "mean"
print(f"Pooling: {pooling}")

# Derive input_dim from directory name
# NOTE: _truth3b must be checked before _truth3 (substring collision)
if "_truth4" in WEIGHTS_DIR:
    input_dim = 5   # 1 (log10 n_hits) + 4 one-hot pdg_label classes
elif "_truth3bprob" in WEIGHTS_DIR:
    input_dim = 4   # 1 (log10 n_hits) + 3 softmax probs [P(other_mu), P(sec_e), P(pEM)]
elif "_truth3b" in WEIGHTS_DIR:
    input_dim = 4   # 1 (log10 n_hits) + 3 one-hot (other+mu / secondary_e / primary_EM_e)
elif "_truth3" in WEIGHTS_DIR:
    input_dim = 4   # 1 (log10 n_hits) + 3 one-hot y classes (other / EM-merged / muon)
elif "_truth2" in WEIGHTS_DIR:
    input_dim = 3   # 1 (log10 n_hits) + 2 binary one-hot (not_primary_EM / primary_EM_e)
elif "_faserglobal" in WEIGHTS_DIR:
    input_dim = 6   # 1 (log10 n_hits) + 5 global FASER features broadcast to each node
elif "_vertexdist" in WEIGHTS_DIR:
    input_dim = 2   # 1 (log10 n_hits) + 1 (distance to true neutrino vertex)
elif "_binaryprob" in WEIGHTS_DIR:
    input_dim = 3   # 1 (log10 n_hits) + 2 binary softmax probs [P(bg), P(pEM)]
elif "_embedding" in WEIGHTS_DIR:
    input_dim = 57  # 1 (log10 n_hits) + 56-dim classifier node embedding
elif "_prob" in WEIGHTS_DIR:
    input_dim = 5   # 1 (log10 n_hits) + 4 softmax probs from 4-class node classifier
else:
    input_dim = 1   # baseline: log10(n_hits) only
print(f"input_dim: {input_dim}")

# Derive beta_loss from checkpoint
beta_loss = checkpoint.get("beta_loss", False)
print(f"Beta loss: {beta_loss}")

# Derive use_faser from directory name
use_faser = "_nofaser" not in WEIGHTS_DIR
print(f"use_faser: {use_faser}")

# Reconstruct model - args must match training exactly
model = NeutrinoGravNetRegressionFASER(
    input_dim=input_dim,
    num_targets=2,    # log10(E_nu), logit(y = E_roe/E_nu) = Bjorken inelasticity
    faser_dim=5,      # nhits_0, nhits_1, nhits_2, faser_x, faser_y
    pooling=pooling,
    beta_loss=beta_loss,
    use_faser=use_faser,
)
model.load_state_dict(checkpoint["model_state_dict"])
model.to(device)
model.eval()
print("Model loaded and set to eval mode.")

# Load the same data as training and reproduce the identical val split.
run_str  = ["nue", "num", "nut", "nun"][RUN % 4]
run_path = torch_path / f"{RUN}/pointnetpp_faser_{DATA_TYPE}_events"

# For _prob models, load *_particle_prob.pt files (probs are baked into data.x).
# For _binaryprob models, load base chunks and augment in-memory (same as training).
if "_binaryprob" in WEIGHTS_DIR:
    chunk_files = sorted(run_path.glob(f"{run_str}_*.pt"))
    chunk_files = [f for f in chunk_files
                   if "_particle_prob" not in f.stem and "_binary_prob" not in f.stem]
elif "_prob" in WEIGHTS_DIR:
    chunk_files = sorted(run_path.glob(f"{run_str}_*_particle_prob.pt"))
else:
    chunk_files = sorted(run_path.glob(f"{run_str}_*.pt"))
    chunk_files = [f for f in chunk_files if "_particle_prob" not in f.stem]

chunk_files = [f for f in chunk_files if int(f.stem.split("_")[1]) in set(CHUNKS)]
print(f"Filtered to shards {CHUNKS[0]}\u2013{CHUNKS[-1]}: {len(chunk_files)} chunks")

dataset = []
for f in chunk_files:
    dataset.extend(torch.load(f, weights_only=False))
print(f"Loaded {len(dataset)} events from {len(chunk_files)} chunks.")

_, val_dataset = train_test_split(dataset, test_size=0.2, random_state=42)
print(f"Val set: {len(val_dataset)} events.")

# Augment val_dataset - must match training augmentation exactly
# (not needed for _prob: probabilities already baked into data.x)
# NOTE: _truth3b must be checked before _truth3 (substring collision)
if "_truth4" in WEIGHTS_DIR:
    print("Augmenting val set with one-hot 4-class truth labels (data.pdg_label).")
    for data in val_dataset:
        y_oh = F.one_hot(data.pdg_label, num_classes=4).float()
        data.x = torch.cat([data.x, y_oh], dim=1)
elif "_truth3bprob" in WEIGHTS_DIR:
    if TRUTH3B_CLASSIFIER_WEIGHTS is None:
        raise ValueError("Set TRUTH3B_CLASSIFIER_WEIGHTS in the config cell.")
    print(f"Augmenting val set with truth3b classifier probs from: {TRUTH3B_CLASSIFIER_WEIGHTS}")
    _clf = NeutrinoGravNetNodesFaser(input_dim=1, num_node_classes=3, faser_dim=5)
    _ckpt_clf = torch.load(TRUTH3B_CLASSIFIER_WEIGHTS, map_location=device, weights_only=False)
    _clf.load_state_dict(_ckpt_clf["model_state_dict"])
    _clf.to(device).eval()
    with torch.no_grad():
        _aug_loader = DataLoader(val_dataset, batch_size=64, shuffle=False)
        _all_probs  = []
        for _batch in _aug_loader:
            _batch = _batch.to(device)
            _out   = _clf(_batch.x, _batch.pos, _batch.batch, _batch.x_faser)
            _probs = torch.softmax(_out, dim=1).cpu()
            _bc    = _batch.batch.cpu()
            for _g in range(int(_bc.max().item()) + 1):
                _all_probs.append(_probs[_bc == _g])
    for _data, _prob in zip(val_dataset, _all_probs):
        _data.x = torch.cat([_data.x, _prob], dim=1)
    del _clf
    torch.cuda.empty_cache()
    print("truth3b classifier augmentation complete.")
elif "_truth3b" in WEIGHTS_DIR:
    print("Augmenting val set with new 3-class truth labels (other+mu / secondary_e / primary_EM_e).")
    remap = torch.tensor([0, 1, 2, 0])
    for data in val_dataset:
        new3_label = remap[data.pdg_label]
        y_oh = F.one_hot(new3_label, num_classes=3).float()
        data.x = torch.cat([data.x, y_oh], dim=1)
elif "_truth3" in WEIGHTS_DIR:
    print("Augmenting val set with one-hot 3-class truth labels (data.y).")
    for data in val_dataset:
        y_oh = F.one_hot(data.y, num_classes=3).float()
        data.x = torch.cat([data.x, y_oh], dim=1)
elif "_truth2" in WEIGHTS_DIR:
    print("Augmenting val set with binary truth label (not_primary_EM / primary_EM_e).")
    remap = torch.tensor([0, 0, 1, 0])
    for data in val_dataset:
        binary_label = remap[data.pdg_label]
        y_oh = F.one_hot(binary_label, num_classes=2).float()
        data.x = torch.cat([data.x, y_oh], dim=1)
elif "_embedding" in WEIGHTS_DIR:
    if BINARY_CLASSIFIER_WEIGHTS is None:
        raise ValueError("Set BINARY_CLASSIFIER_WEIGHTS in the config cell to the classifier used for embedding.")
    print(f"Augmenting val set with classifier node embedding from: {BINARY_CLASSIFIER_WEIGHTS}")
    _ckpt_clf = torch.load(BINARY_CLASSIFIER_WEIGHTS, map_location=device, weights_only=False)
    _n_cls = _ckpt_clf["num_node_classes"]
    _cfg   = _ckpt_clf.get("model_config", {})
    _clf = NeutrinoGravNetNodesFaser(
        input_dim=1, num_node_classes=_n_cls, faser_dim=5,
        n_gravstack=_cfg.get("n_gravstack", 3),
        out_channels=_cfg.get("out_channels", 16),
        n_feature_transform=_cfg.get("n_feature_transform", 16),
        k=_cfg.get("k", 12),
    )
    _clf.load_state_dict(_ckpt_clf["model_state_dict"])
    _clf.to(device).eval()
    with torch.no_grad():
        _aug_loader = DataLoader(val_dataset, batch_size=64, shuffle=False)
        _all_embs   = []
        for _batch in _aug_loader:
            _batch = _batch.to(device)
            _, _emb = _clf(_batch.x, _batch.pos, _batch.batch,
                           _batch.x_faser, return_embedding=True)
            _emb = _emb.cpu()
            _bc  = _batch.batch.cpu()
            for _g in range(int(_bc.max().item()) + 1):
                _all_embs.append(_emb[_bc == _g])
    _emb = _all_embs[0]  # for emb_dim print below
    for _data, _emb_g in zip(val_dataset, _all_embs):
        _data.x = torch.cat([_data.x, _emb_g], dim=1)
    del _clf
    torch.cuda.empty_cache()
    print(f"Classifier embedding augmentation complete (emb_dim={_emb.shape[1]}).")
elif "_vertexdist" in WEIGHTS_DIR:
    print("Augmenting val set with ground-truth vertex distance.")
    for data in val_dataset:
        vertex_dist = torch.norm(
            data.pos - data.true_pos_centered.unsqueeze(0), dim=1, keepdim=True
        )
        data.x = torch.cat([data.x, vertex_dist], dim=1)
    print("Vertex distance augmentation complete.")
elif "_binaryprob" in WEIGHTS_DIR:
    if BINARY_CLASSIFIER_WEIGHTS is None:
        raise ValueError("Set BINARY_CLASSIFIER_WEIGHTS in the config cell to the binary classifier best_model.pt path.")
    print(f"Augmenting val set with binary classifier probs from: {BINARY_CLASSIFIER_WEIGHTS}")
    _clf = NeutrinoGravNetNodesFaser(input_dim=1, num_node_classes=2, faser_dim=5)
    _ckpt_clf = torch.load(BINARY_CLASSIFIER_WEIGHTS, map_location=device, weights_only=False)
    _clf.load_state_dict(_ckpt_clf["model_state_dict"])
    _clf.to(device).eval()
    with torch.no_grad():
        _aug_loader = DataLoader(val_dataset, batch_size=64, shuffle=False)
        _all_probs  = []
        for _batch in _aug_loader:
            _batch = _batch.to(device)
            _out   = _clf(_batch.x, _batch.pos, _batch.batch, _batch.x_faser)
            _probs = torch.softmax(_out, dim=1).cpu()
            _bc    = _batch.batch.cpu()
            for _g in range(int(_bc.max().item()) + 1):
                _all_probs.append(_probs[_bc == _g])
    for _data, _prob in zip(val_dataset, _all_probs):
        _data.x = torch.cat([_data.x, _prob], dim=1)
    del _clf
    torch.cuda.empty_cache()
    print("Binary classifier augmentation complete.")
elif "_faserglobal" in WEIGHTS_DIR:
    print("Augmenting val set with global FASER features (injected to each node).")
    for data in val_dataset:
        faser_expanded = data.x_faser.unsqueeze(0).expand(data.num_nodes, -1)
        data.x = torch.cat([data.x, faser_expanded], dim=1)

val_loader = DataLoader(val_dataset, batch_size=32, shuffle=False)

# Inference cache key - encodes the specific checkpoint
cache_key  = f"ep{checkpoint['epoch']}_vl{checkpoint['val_loss']:.6f}"
cache_path = weights_path / f"inference_cache_{cache_key}.npz"

**Inference**

Model outputs are `(log10(E_nu), logit(y))` where `y = E_roe / E_nu` (Bjorken inelasticity). Physical energies are recovered as:

```
E_nu     = 10 ** pred[:,0]
y        = sigmoid(pred[:,1])
E_roe    = y * E_nu
E_lepton = (1 - y) * E_nu
```

Energy conservation `E_lepton + E_roe = E_nu` holds exactly by construction. Targets in the dataset are raw linear TeV.

In [ ]:
def preds_to_physical(preds, norm_stats=None, beta_loss=False):
    """
    Convert model outputs to physical energies (TeV).

    If norm_stats is provided (new standardised models), outputs are first
    destandardised back to reparam space before converting to physical units.
    If norm_stats is None (old inverse-variance-weighted models), outputs are
    assumed to already be in reparam space (backwards compatible).

    preds: [N, 2] or [N, 3] torch tensor
      beta_loss=False: col 0: log10(E_nu) [std], col 1: logit(y) [std]
      beta_loss=True:  col 0: log10(E_nu) [std], col 1: alpha, col 2: beta (softplus-activated)
    Returns [N, 3] numpy array - (E_nu, E_lepton, E_roe) in TeV
    """
    if norm_stats is not None:
        log_E_nu = preds[:, 0] * norm_stats["sigma_enu"] + norm_stats["mu_enu"]
    else:
        log_E_nu = preds[:, 0]
    E_nu = 10 ** log_E_nu
    if beta_loss:
        alpha, beta_p = preds[:, 1], preds[:, 2]
        y = alpha / (alpha + beta_p)           # Beta distribution mean
    else:
        if norm_stats is not None:
            logit_y = preds[:, 1] * norm_stats["sigma_logit"] + norm_stats["mu_logit"]
        else:
            logit_y = preds[:, 1]
        y = torch.sigmoid(logit_y)
    E_roe = y * E_nu          # y = inelasticity = E_roe/E_nu
    E_lep = (1 - y) * E_nu
    return torch.stack([E_nu, E_lep, E_roe], dim=1).numpy()


norm_stats = checkpoint.get("norm_stats", None)
if norm_stats is not None:
    print(f"Standardised model - norm_stats loaded from checkpoint:")
    print(f"  log10(E_nu): mu={norm_stats['mu_enu']:.4f}, sigma={norm_stats['sigma_enu']:.4f}")
    print(f"  logit(y):    mu={norm_stats['mu_logit']:.4f}, sigma={norm_stats['sigma_logit']:.4f}")
else:
    print("Non-standardised model (old inverse-variance weighting) - no destandardisation applied.")

if cache_path.exists():
    print(f"Loading inference from cache: {cache_path.name}")
    _c = np.load(cache_path)
    preds_raw      = torch.from_numpy(_c['preds_raw'])
    targets_linear = _c['targets_linear']
    preds_linear   = preds_to_physical(preds_raw, norm_stats, beta_loss=beta_loss)
    print(f"Loaded {len(preds_linear)} events.")
else:
    all_preds   = []
    all_targets = []

    with torch.no_grad():
        for data in val_loader:
            data    = data.to(device)
            raw     = model(data.x, data.pos, data.batch, data.x_faser)  # [B, 2]
            targets = torch.stack([data.E_nu, data.E_lepton, data.E_roe], dim=1)  # [B, 3] linear TeV
            all_preds.append(raw.cpu())
            all_targets.append(targets.cpu())

    preds_raw      = torch.cat(all_preds)             # [N, 2]  raw model output
    targets_linear = torch.cat(all_targets).numpy()   # [N, 3]  TeV
    preds_linear   = preds_to_physical(preds_raw, norm_stats, beta_loss=beta_loss)  # [N, 3]  TeV

    np.savez_compressed(cache_path,
                        preds_raw=preds_raw.numpy(),
                        targets_linear=targets_linear)
    print(f"Saved inference cache: {cache_path.name}")

print(f"Inference done: {len(preds_linear)} events")
for i, name in enumerate(TARGET_NAMES):
    print(f"  {name:12s}: true [{targets_linear[:, i].min():.4f}, "
          f"{targets_linear[:, i].max():.4f}] TeV")

In [ ]:
# Physical distributions
fig, axes = plt.subplots(1, 3, figsize=(15, 4))
for i, (name, latex) in enumerate(zip(TARGET_NAMES, TARGET_LATEX)):
    axes[i].hist(targets_linear[:, i], bins=50, histtype="step", linewidth=1.5)
    axes[i].set_xlabel(f"True {latex} [TeV]")
    axes[i].set_ylabel("Events")
    axes[i].set_title(name)
plt.suptitle("True energy distributions (physical)", fontsize=11)
plt.tight_layout()
plt.savefig(figures_path / "dist_labels.png", dpi=500, bbox_inches='tight')
plt.show()

if norm_stats is not None:
    # Compute reparametrised and standardised values (val set)
    norm_stats = checkpoint["norm_stats"]
    log_E_nu   = np.log10(targets_linear[:, 0].clip(min=1e-6))
    logit_y    = np.log(targets_linear[:, 1].clip(min=1e-6) / targets_linear[:, 2].clip(min=1e-6))
    z_enu      = (log_E_nu - norm_stats["mu_enu"])   / norm_stats["sigma_enu"]
    z_logit    = (logit_y  - norm_stats["mu_logit"]) / norm_stats["sigma_logit"]

    # Train set stats
    train_dataset, _ = train_test_split(dataset, test_size=0.2, random_state=42)
    log_E_nu_train   = np.array([np.log10(max(d.E_nu.item(), 1e-6)) for d in train_dataset])
    logit_y_train    = np.array([np.log(max(d.E_roe.item(), 1e-6) / max(d.E_lepton.item(), 1e-6)) for d in train_dataset])
    z_enu_train      = (log_E_nu_train - norm_stats["mu_enu"])   / norm_stats["sigma_enu"]
    z_logit_train    = (logit_y_train  - norm_stats["mu_logit"]) / norm_stats["sigma_logit"]

    print("Train set:")
    print(f"  log10(E_nu): [{log_E_nu_train.min():.2f}, {log_E_nu_train.max():.2f}]  std = {log_E_nu_train.std():.3f}   z std = {z_enu_train.std():.3f}")
    print(f"  logit(y):    [{logit_y_train.min():.2f}, {logit_y_train.max():.2f}]  std = {logit_y_train.std():.3f}   z std = {z_logit_train.std():.3f}")
    print(f"  variance ratio: {logit_y_train.std()**2 / log_E_nu_train.std()**2:.1f}x")
    print()
    print("Val set:")
    print(f"  log10(E_nu): [{log_E_nu.min():.2f}, {log_E_nu.max():.2f}]  std = {log_E_nu.std():.3f}   z std = {z_enu.std():.3f}")
    print(f"  logit(y):    [{logit_y.min():.2f}, {logit_y.max():.2f}]  std = {logit_y.std():.3f}   z std = {z_logit.std():.3f}")
    print(f"  variance ratio: {logit_y.std()**2 / log_E_nu.std()**2:.1f}x")

    # Reparametrised and standardised plots
    fig, axes = plt.subplots(1, 4, figsize=(18, 4))

    axes[0].hist(log_E_nu, bins=50, histtype="step", linewidth=1.5)
    axes[0].set_xlabel(r"$\log_{10}(E_\nu)$")
    axes[0].set_ylabel("Events")
    axes[0].set_title(r"$\log_{10}(E_\nu)$  [reparam]")

    axes[1].hist(z_enu, bins=50, histtype="step", linewidth=1.5)
    axes[1].set_xlabel(r"$z_{\log E_\nu}$")
    axes[1].set_ylabel("Events")
    axes[1].set_title(r"$\log_{10}(E_\nu)$  [standardised]")

    axes[2].hist(logit_y, bins=50, histtype="step", linewidth=1.5)
    axes[2].set_xlabel(r"$\mathrm{logit}(y)$")
    axes[2].set_ylabel("Events")
    axes[2].set_title(r"$\mathrm{logit}(y)$  [reparam]")

    axes[3].hist(z_logit, bins=50, histtype="step", linewidth=1.5)
    axes[3].set_xlabel(r"$z_{\mathrm{logit}(y)}$")
    axes[3].set_ylabel("Events")
    axes[3].set_title(r"$\mathrm{logit}(y)$  [standardised]")

    plt.suptitle("Reparametrised and standardised target distributions", fontsize=11)
    plt.tight_layout()
    plt.savefig(figures_path / "reparam_std.png", dpi=500, bbox_inches='tight')
    plt.show()



**True vs reconstructed energy**

In [ ]:
import matplotlib.colors as mcolors

fig, axes = plt.subplots(1, 3, figsize=(18, 6))

for i, (name, latex) in enumerate(zip(TARGET_NAMES, TARGET_LATEX)):
    ax     = axes[i]
    y_true = targets_linear[:, i]
    y_pred = preds_linear[:, i]

    # Metrics
    ss_res    = np.sum((y_true - y_pred) ** 2)
    ss_tot    = np.sum((y_true - y_true.mean()) ** 2)
    r2        = 1 - ss_res / ss_tot
    pearson_r = np.corrcoef(y_true, y_pred)[0, 1]

    # Filter to positive values before log-scale hexbin
    pos    = (y_true > 0) & (y_pred > 0)
    yt_pos = y_true[pos]
    yp_pos = y_pred[pos]

    # Axis range from true values only - keeps the plot square and clips prediction outliers
    lo = yt_pos.min()
    hi = yt_pos.max()

    cmap = mcolors.LinearSegmentedColormap.from_list(
    'trunc', plt.cm.ocean(np.linspace(0.3, 0.9, 100)))
    ax.set_facecolor("#F5F5F5")
    hb = ax.hexbin(yt_pos, yp_pos,
                   xscale='log', yscale='log',
                   gridsize=120, cmap=cmap, bins='log',
                   mincnt=1, alpha=0.8,
                   extent=[np.log10(lo), np.log10(hi),
                           np.log10(lo), np.log10(hi)])
    plt.colorbar(hb, ax=ax, label='Counts')

    ax.set_xlim(lo, hi)
    ax.set_ylim(lo, hi)
    ax.set_aspect('equal')

    ax.plot([lo, hi], [lo, hi], "k--", linewidth=0.5, label="y = x")
    ax.set_xlabel(rf"True {latex} [{TARGET_UNITS}]")
    ax.set_ylabel(rf"Reconstructed {latex} [{TARGET_UNITS}]")
    ax.legend(fontsize=8, frameon=False)
    ax.set_title(name)
    ax.text(
        0.05, 0.95,
        f"$R^2$ = {r2:.4f}\n$r$ = {pearson_r:.4f}",
        transform=ax.transAxes, va="top", fontsize=9,
        bbox=dict(facecolor='white', alpha=0.8, edgecolor='none'),
    )
    ax.spines[['top', 'right']].set_visible(False)
    ax.grid(True, linestyle=':', linewidth=0.8, color='gray', alpha=0.3)

plt.tight_layout()
plt.savefig(figures_path / "true_vs_reco.png", dpi=500, bbox_inches='tight')
plt.show()
print("Saved: true_vs_reco.png")
print()
print(f"{'Target':<12} {'R²':>10} {'Pearson r':>12}")
for i, name in enumerate(TARGET_NAMES):
    y_true = targets_linear[:, i]
    y_pred = preds_linear[:, i]
    r2 = 1 - np.sum((y_true - y_pred)**2) / np.sum((y_true - y_true.mean())**2)
    r  = np.corrcoef(y_true, y_pred)[0, 1]
    print(f"{name:<12} {r2:>10.4f} {r:>12.4f}")


**Resolution histograms per energy bin**


For each energy bin, compute $(p_\mathrm{pred} - p_\mathrm{true}) / p_\mathrm{true}$ for every event in that slice, giving a distribution of relative errors. A Gaussian is then fit to this distribution:

- **$\mu$**: centre of the distribution - if $\mu = +0.1$, the model is systematically 10% too high in that energy slice
- **$\sigma$**: width of the distribution - how spread out errors are around that centre

These are independent:

| $\mu$ | $\sigma$ | Interpretation |
|-------|----------|----------------|
| 0     | 0.4      | Unbiased but imprecise - errors scatter symmetrically around zero by ±40% |
| 0.4   | 0.1      | Precise but systematically wrong - consistently ~40% too high, little scatter |
| 0.2   | 0.4      | Both biased and imprecise |

For a Gaussian, $\sigma$ is the **68% interval**: ~68% of events in that bin have relative errors within $\pm\sigma$ of $\mu$, ~95% within $\pm 2\sigma$.

So $\sigma = 0.4$ for $E_\mathrm{roe}$ in a bin means 68% of events in that energy range have a relative prediction error within ±40%.


In [ ]:
def gauss(x, A, mu, sigma):
    return A * np.exp(-0.5 * ((x - mu) / sigma) ** 2)

for target_idx, (name, latex, col) in enumerate(zip(TARGET_NAMES, TARGET_LATEX, [C1, C2, C3])):
    y_true     = targets_linear[:, target_idx]
    y_pred     = preds_linear[:, target_idx]
    resolution = (y_pred - y_true) / y_true

    fig, axes_grid = plt.subplots(3, 3, figsize=(12, 10), constrained_layout=True)
    axes_flat = axes_grid.flatten()

    for j, (emin, emax) in enumerate(ENERGY_BINS_TEV):
        if j >= 9:
            break
        ax      = axes_flat[j]
        mask    = (y_true >= emin) & (y_true < emax)
        res_bin = resolution[mask]

        if len(res_bin) == 0:
            ax.text(0.5, 0.5, "No events", ha="center", va="center",
                    transform=ax.transAxes)
            ax.set_title(f"{emin}-{emax} TeV")
            continue

        p1, p99  = np.percentile(res_bin, [1, 99])
        xlim     = min(max(abs(p1), abs(p99), 0.1), 5.0)
        bins_arr = np.linspace(-xlim, xlim, 60)

        counts, bin_edges, _ = ax.hist(
            res_bin, bins=bins_arr,
            histtype='stepfilled', alpha=0.6, color=col, linewidth=0,
        )

        bin_centers_fit = 0.5 * (bin_edges[:-1] + bin_edges[1:])
        try:
            popt_g, _ = curve_fit(
                gauss, bin_centers_fit, counts,
                p0=[counts.max(), np.mean(res_bin), np.std(res_bin)],
                maxfev=5000,
            )
            sigma_fit = abs(popt_g[2])
            x_gauss   = np.linspace(-xlim, xlim, 300)
            ax.plot(x_gauss, gauss(x_gauss, *popt_g),
                    color=col, linewidth=1.5, alpha=1.0, label="Gauss fit")
            label_text = f"mu = {popt_g[1]:.3f}\nsigma = {sigma_fit:.3f}"
            ax.legend(fontsize=7, frameon=False)
        except (RuntimeError, ValueError):
            label_text = f"mean = {np.mean(res_bin):.3f}\nstd  = {np.std(res_bin):.3f}"

        ax.set_title(f"{emin}-{emax} TeV  (N={len(res_bin)})")
        ax.set_xlabel("(pred - true) / true")
        ax.set_ylabel("Events")
        ax.set_xlim(-xlim, xlim)
        ax.text(
            0.05, 0.95, label_text,
            transform=ax.transAxes, va="top", fontsize=9,
            bbox=dict(facecolor='white', alpha=0.7, edgecolor='none'),
        )
        ax.spines[['top', 'right']].set_visible(False)
        ax.grid(True, linestyle=':', linewidth=0.8, color='gray', alpha=0.3)

    fig.suptitle(f"{name} resolution per energy bin", fontsize=14)
    plt.savefig(figures_path / f"{name}_ResolutionPerBin.png", dpi=500, bbox_inches='tight')
    plt.show()
    print(f"Saved: {name}_ResolutionPerBin.png")

**Resolution vs energy**

std of (pred − true) / true in each energy bin. Should decrease with energy as higher-energy showers produce more hits and are easier to reconstruct.

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(18, 5))

for target_idx, (name, latex, col) in enumerate(zip(TARGET_NAMES, TARGET_LATEX, [C1, C2, C3])):
    ax         = axes[target_idx]
    y_true     = targets_linear[:, target_idx]
    y_pred     = preds_linear[:, target_idx]
    resolution = (y_pred - y_true) / y_true

    bin_centers = []
    sigmas      = []
    sigma_errs  = []

    for emin, emax in ENERGY_BINS_TEV:
        mask    = (y_true >= emin) & (y_true < emax)
        res_bin = resolution[mask]

        if len(res_bin) < 50:   # skip underpopulated bins
            continue

        sigma     = np.std(res_bin)
        sigma_err = sigma / np.sqrt(2 * len(res_bin))
        center    = np.mean(y_true[mask])

        bin_centers.append(center)
        sigmas.append(sigma)
        sigma_errs.append(sigma_err)

    if bin_centers:
        ax.errorbar(bin_centers, sigmas, yerr=sigma_errs,
                    fmt='o', markersize=5, capsize=3, linewidth=0.8,
                    color=col, markerfacecolor='white', markeredgewidth=1.2)

    for x, s in zip(bin_centers, sigmas):
        if s >= 1.0:
            ax.annotate("", xy=(x, 1.0), xytext=(x, 0.92),
                        arrowprops=dict(arrowstyle="->", linewidth=1.5, color="red"))

    ax.set_xscale("log")
    ax.set_ylim(0, 1.05)
    ax.axhline(1.0, color="gray", linestyle="--", linewidth=0.8, alpha=0.5)
    ax.set_xlabel(rf"True {latex} [{TARGET_UNITS}]")
    ax.set_ylabel(rf"{latex} Resolution ($\sigma$)")
    ax.set_title(name)
    ax.spines[['top', 'right']].set_visible(False)
    ax.grid(True, linestyle=':', linewidth=0.8, color='gray', alpha=0.3)

plt.tight_layout()
plt.savefig(figures_path / "resolution_vs_true.png", dpi=500, bbox_inches='tight')
plt.show()
print("Saved: resolution_vs_true.png")

**Bias vs energy**

Mean of (pred − true) / true per energy bin. Should be near zero. A non-zero slope means the model systematically over- or under-estimates at certain energies.

In [ ]:
def line_log(x, m, c):
    """Bias model: linear in log10(E)."""
    return m * np.log10(x) + c

fig, axes   = plt.subplots(1, 3, figsize=(18, 5))
bias_params = {}

for target_idx, (name, latex, col) in enumerate(zip(TARGET_NAMES, TARGET_LATEX, [C1, C2, C3])):
    ax         = axes[target_idx]
    y_true     = targets_linear[:, target_idx]
    y_pred     = preds_linear[:, target_idx]
    resolution = (y_pred - y_true) / y_true

    bin_centers = []
    means       = []
    mean_errs   = []
    ns          = []

    for emin, emax in ENERGY_BINS_TEV:
        mask    = (y_true >= emin) & (y_true < emax)
        res_bin = resolution[mask]
        n       = len(res_bin)
        if n < 50:
            continue
        sigma = np.std(res_bin)
        bin_centers.append(np.mean(y_true[mask]))
        means.append(np.mean(res_bin))
        mean_errs.append(sigma / np.sqrt(n))
        ns.append(n)

    if bin_centers:
        ax.errorbar(bin_centers, means, yerr=mean_errs,
                    fmt='o', markersize=5, capsize=3, linewidth=0.8,
                    color=col, markerfacecolor='white', markeredgewidth=1.2)

    if len(bin_centers) >= 2:
        try:
            popt, _ = curve_fit(
                line_log, bin_centers, means,
                sigma=mean_errs, absolute_sigma=True,
            )
            x_fit = np.logspace(
                np.log10(min(bin_centers)), np.log10(max(bin_centers)), 100
            )
            ax.plot(x_fit, line_log(x_fit, *popt),
                    color=col, linestyle='--', linewidth=1.0, alpha=0.8,
                    label=f"fit: {popt[0]:.2e} · log₁₀E + {popt[1]:.2e}")
            ax.legend(fontsize=8, frameon=False)
            bias_params[name] = popt
        except RuntimeError:
            pass

    ax.axhline(0, color="k", linestyle=":", linewidth=0.8, alpha=0.5)
    ax.set_xscale("log")
    ax.set_xlabel(f"True {latex} [{TARGET_UNITS}]")
    ax.set_ylabel("Bias  (mean ± std/√N)")
    ax.set_title(name)
    ax.spines[['top', 'right']].set_visible(False)
    ax.grid(True, linestyle=':', linewidth=0.8, color='gray', alpha=0.3)

plt.tight_layout()
plt.savefig(figures_path / "bias.png", dpi=500, bbox_inches='tight')
plt.show()
print("Saved: bias.png")

**Summary**

In [ ]:
print(f"Best model : epoch {checkpoint['epoch'] + 1}")
print(f"Val MSE    : {checkpoint['val_loss']:.4f}")
print(f"Val RMSE   : {checkpoint['val_rmse']:.4f}")
print()
print(f"{'Target':<12} {'Median RelErr':>14} {'Std(res)':>10} {'Bias (mean)':>12}")

for i, name in enumerate(TARGET_NAMES):
    y_true  = targets_linear[:, i]
    y_pred  = preds_linear[:, i]
    rel_err = np.abs(y_pred - y_true) / y_true
    res     = (y_pred - y_true) / y_true
    print(f"{name:<12} {np.median(rel_err):>14.3f} {res.std():>10.3f} {res.mean():>+12.3f}")

print()
print(f"Figures: {figures_path}")

In [ ]:
# ── Bootstrap confidence intervals on headline metrics ───────────────────────
# Resamples the val set 1000× (with replacement) to get 95% CIs on σ, R², bias.
# Same indices used for y_true and y_pred → CIs reflect val-set variability only.

def bootstrap_metric(y_true, y_pred, metric_fn, n_boot=1000, ci=95, seed=42):
    rng = np.random.default_rng(seed)
    n = len(y_true)
    stats = []
    for _ in range(n_boot):
        idx = rng.integers(0, n, n)
        stats.append(metric_fn(y_true[idx], y_pred[idx]))
    lo, hi = np.percentile(stats, [(100 - ci) / 2, 100 - (100 - ci) / 2])
    return float(np.median(stats)), float(lo), float(hi)

def resolution_fn(yt, yp):
    return float(np.std((yp - yt) / yt))

def r2_fn(yt, yp):
    return float(1 - np.sum((yt - yp) ** 2) / np.sum((yt - yt.mean()) ** 2))

def bias_fn(yt, yp):
    return float(np.mean((yp - yt) / yt))

print(f"Bootstrap 95% CIs on headline metrics  (n_boot=1000, n_val={len(preds_linear)})\n")
hdr = f"{'Target':<12}  {'σ':>6}  {'95% CI':^15}    {'R²':>7}  {'95% CI':^15}    {'bias':>7}  {'95% CI':^15}"
print(hdr)
print("─" * len(hdr))
for i, name in enumerate(TARGET_NAMES):
    yt = targets_linear[:, i]
    yp = preds_linear[:, i]
    s,  sl, sh = bootstrap_metric(yt, yp, resolution_fn)
    r,  rl, rh = bootstrap_metric(yt, yp, r2_fn)
    b,  bl, bh = bootstrap_metric(yt, yp, bias_fn)
    print(f"{name:<12}  {s:>6.3f}  [{sl:.3f}, {sh:.3f}]    "
          f"{r:>7.4f}  [{rl:.4f}, {rh:.4f}]    "
          f"{b:>+7.4f}  [{bl:+.4f}, {bh:+.4f}]")


**Baseline: total calorimeter hits**

Each graph node stores log10(n_hits) for one calorimeter super-pixel. Summing 10^x over all nodes gives total raw hits - the same quantity as `len(hit_colID)` in `explore_tau_dataset.ipynb`, which correlates strongly with E_nu. A linear fit `E = a * total_hits + b` is the simplest possible reconstruction. GravNet should beat it, especially for E_lep and E_roe where the spatial shower structure matters.

In [ ]:
# ── Total calorimeter hits per event ─────────────────────────────────────────
hits_list = []
for data in val_dataset:
    hits_list.append((10 ** data.x).sum().item())

total_hits_arr = np.array(hits_list)

# ── Linear fit for each target: E_target = a * total_hits + b ────────────────
coeffs_all       = []
linear_preds_all = []
for i in range(3):
    c = np.polyfit(total_hits_arr, targets_linear[:, i], 1)
    coeffs_all.append(c)
    linear_preds_all.append(np.polyval(c, total_hits_arr))

linear_res_all  = [
    (linear_preds_all[i] - targets_linear[:, i]) / targets_linear[:, i]
    for i in range(3)
]
gravnet_res_all = [
    (preds_linear[:, i] - targets_linear[:, i]) / targets_linear[:, i]
    for i in range(3)
]

# ── Figure 1: Hits scatter + fit ──────────────────────────────────────────────
fig, axes = plt.subplots(1, 3, figsize=(18, 5))
for i, (name, latex, col) in enumerate(zip(TARGET_NAMES, TARGET_LATEX, [C1, C2, C3])):
    ax = axes[i]
    r  = np.corrcoef(total_hits_arr, targets_linear[:, i])[0, 1]
    ax.scatter(total_hits_arr, targets_linear[:, i],
               s=4, alpha=0.4, color=col, linewidths=0)
    x_fit = np.linspace(total_hits_arr.min(), total_hits_arr.max(), 100)
    ax.plot(x_fit, np.polyval(coeffs_all[i], x_fit),
            color='k', linestyle='--', linewidth=1.2, label=f"r = {r:.3f}")
    ax.set_xlabel("Total calorimeter hits")
    ax.set_ylabel(rf"True {latex} [TeV]")
    ax.set_title(name)
    ax.legend(fontsize=9, frameon=False)
    ax.spines[['top', 'right']].set_visible(False)
    ax.grid(True, linestyle=':', linewidth=0.8, color='gray', alpha=0.3)
plt.suptitle("Baseline: total calorimeter hits vs energy targets", fontsize=12)
plt.tight_layout()
plt.savefig(figures_path / "baseline_scatter.png", dpi=500, bbox_inches='tight')
plt.show()
print("Saved: baseline_scatter.png")

# ── Figure 2: Grouped bar — overall resolution σ(E)/E per target ─────────────
gn_sigmas  = [np.std(gravnet_res_all[i])  for i in range(3)]
lin_sigmas = [np.std(linear_res_all[i])   for i in range(3)]

x     = np.arange(3)
width = 0.35
fig, ax = plt.subplots(figsize=(3.5, 2.8))
bars_gn  = ax.bar(x - width/2, gn_sigmas,  width, label="GravNet",    color=C1, alpha=0.85)
bars_lin = ax.bar(x + width/2, lin_sigmas, width, label="Linear fit", color=C2, alpha=0.85)

# Annotate σ values above each bar
for bar in bars_gn:
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.003,
            f"{bar.get_height():.3f}", ha='center', va='bottom', fontsize=8)
for bar in bars_lin:
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.003,
            f"{bar.get_height():.3f}", ha='center', va='bottom', fontsize=8)

# Annotate % improvement above each bar pair
for i in range(3):
    gn_s  = gn_sigmas[i]
    lin_s = lin_sigmas[i]
    pct   = (lin_s - gn_s) / lin_s * 100
    pair_x = x[i]
    pair_y = max(lin_s, gn_s) + 0.04
    ax.text(pair_x, pair_y, f"−{pct:.1f}%",
            ha='center', va='bottom', fontsize=7, color='#2a6e3f',
            fontweight='bold')

ax.set_xticks(x)
ax.set_xticklabels(TARGET_LATEX, fontsize=9)
ax.set_ylabel(r"Resolution $\sigma(E)/E$", fontsize=9)
ax.tick_params(labelsize=8)
ax.legend(frameon=False, fontsize=8)
ax.spines[['top', 'right']].set_visible(False)
ax.set_ylim(0, max(lin_sigmas) * 1.28)
plt.tight_layout()
plt.savefig(figures_path / "gravnet_vs_linear.png", dpi=300, bbox_inches='tight')
plt.show()
print("Saved: gravnet_vs_linear.png")

# ── Summary ───────────────────────────────────────────────────────────────────
print(f"\n{'Target':<12} {'GravNet σ':>12} {'Linear σ':>11} {'GravNet bias':>14} {'Linear bias':>12}")
for i, name in enumerate(TARGET_NAMES):
    print(f"{name:<12} {np.std(gravnet_res_all[i]):>12.3f} {np.std(linear_res_all[i]):>11.3f} "
          f"{np.mean(gravnet_res_all[i]):>+14.3f} {np.mean(linear_res_all[i]):>+12.3f}")


In [ ]:
import json, datetime
from scipy.optimize import curve_fit
from scipy.stats import pearsonr
from sklearn.metrics import r2_score

if not SAVE_EVAL_HISTORY:
    print("SAVE_EVAL_HISTORY=False — skipping. Set to True in the config cell to record stats.")
else:
    def _gauss(x, A, mu, sigma):
        return A * np.exp(-0.5 * ((x - mu) / sigma) ** 2)

    def _per_bin_stats(y_true, y_pred, bins):
        rows = []
        res = (y_pred - y_true) / y_true
        for emin, emax in bins:
            mask = (y_true >= emin) & (y_true < emax)
            n = int(mask.sum())
            if n == 0:
                continue
            r = res[mask]
            row = {"emin": emin, "emax": emax, "n": n,
                   "mean_res": float(np.mean(r)), "std_res": float(np.std(r))}
            p1, p99 = np.percentile(r, [1, 99])
            xlim = min(max(abs(p1), abs(p99), 0.1), 5.0)
            counts, edges = np.histogram(r, bins=np.linspace(-xlim, xlim, 60))
            centers = 0.5 * (edges[:-1] + edges[1:])
            try:
                popt, _ = curve_fit(_gauss, centers, counts,
                                    p0=[counts.max(), np.mean(r), np.std(r)], maxfev=5000)
                row["gauss_mu"]    = float(popt[1])
                row["gauss_sigma"] = float(abs(popt[2]))
            except (RuntimeError, ValueError):
                row["gauss_mu"]    = None
                row["gauss_sigma"] = None
            rows.append(row)
        return rows

    # ── Collect summary ───────────────────────────────────────────────────────
    summary = {
        "timestamp":   datetime.datetime.now().isoformat(),
        "weights_dir": WEIGHTS_DIR,
        "cache_key":   cache_key,
        "checkpoint": {
            "epoch":          int(checkpoint["epoch"]),
            "loss_fn":        checkpoint.get("loss_fn"),
            "huber_delta":    checkpoint.get("huber_delta"),
            "val_loss":       float(checkpoint["val_loss"]),
            "val_rmse":       float(checkpoint["val_rmse"]),
            "parametrisation": checkpoint.get("parametrisation"),
        },
        "n_val_events": int(len(preds_linear)),
        "overall":     {},
        "per_bin":     {},
        "inelasticity": {},
        "baseline":    {},
    }

    for i, name in enumerate(TARGET_NAMES):
        yt = targets_linear[:, i]
        yp = preds_linear[:, i]
        res = (yp - yt) / yt
        r2  = float(r2_score(yt, yp))
        pr, _ = pearsonr(yt, yp)
        summary["overall"][name] = {
            "r2":             r2,
            "pearson_r":      float(pr),
            "median_rel_err": float(np.median(np.abs(res))),
            "mean_rel_err":   float(np.mean(np.abs(res))),
            "std_res":        float(np.std(res)),
            "mean_bias":      float(np.mean(res)),
        }
        summary["per_bin"][name] = _per_bin_stats(yt, yp, ENERGY_BINS_TEV)

    # Inelasticity
    inel_true = targets_linear[:, 2] / targets_linear[:, 0].clip(min=1e-6)  # E_roe/E_nu = inelasticity
    inel_pred = preds_linear[:, 2]   / preds_linear[:, 0].clip(min=1e-6)
    pr_i, _   = pearsonr(inel_true, inel_pred)
    summary["inelasticity"] = {
        "r2":        float(r2_score(inel_true, inel_pred)),
        "pearson_r": float(pr_i),
        "std_res":   float(np.std((inel_pred - inel_true) / inel_true.clip(min=1e-6))),
        "mean_bias": float(np.mean((inel_pred - inel_true) / inel_true.clip(min=1e-6))),
    }

    # Baseline comparison (recompute linear fit)
    hits_list = [float((10 ** data.x).sum()) for data in val_dataset]
    total_hits_arr_s = np.array(hits_list)
    for i, name in enumerate(TARGET_NAMES):
        yt  = targets_linear[:, i]
        yp  = preds_linear[:, i]
        c   = np.polyfit(total_hits_arr_s, yt, 1)
        lin = np.polyval(c, total_hits_arr_s)
        summary["baseline"][name] = {
            "gravnet_std_res": float(np.std((yp - yt) / yt)),
            "linear_std_res":  float(np.std((lin - yt) / yt)),
            "gravnet_r2":      float(r2_score(yt, yp)),
            "linear_r2":       float(r2_score(yt, lin)),
        }

    # ── Append to JSONL ───────────────────────────────────────────────────────
    _history_file = weights_path.parent / "eval_history.jsonl"
    with open(_history_file, "a") as f:
        f.write(json.dumps(summary) + "\n")

    print(f"Eval summary appended to: {_history_file}")
    print(f"Entry timestamp: {summary['timestamp']}")
    print(f"Weights dir:     {WEIGHTS_DIR}")
    print(f"Val events:      {summary['n_val_events']}")
    print()
    print(f"{'Target':<12} {'R²':>8} {'Pearson r':>10} {'Med|RelErr|':>12} {'Std(res)':>10} {'Bias':>8}")
    for name in TARGET_NAMES:
        s = summary["overall"][name]
        print(f"{name:<12} {s['r2']:>8.4f} {s['pearson_r']:>10.4f} "
              f"{s['median_rel_err']:>12.3f} {s['std_res']:>10.3f} {s['mean_bias']:>+8.3f}")
    print()
    print(f"Inelasticity  R²={summary['inelasticity']['r2']:.4f}  "
          f"r={summary['inelasticity']['pearson_r']:.4f}  "
          f"bias={summary['inelasticity']['mean_bias']:+.3f}")
    print()
    print(f"{'Target':<12} {'GravNet σ':>10} {'Linear σ':>10}")
    for name in TARGET_NAMES:
        b = summary["baseline"][name]
        print(f"{name:<12} {b['gravnet_std_res']:>10.3f} {b['linear_std_res']:>10.3f}")

**Lepton energy fraction and energy conservation**

The model's second output represents `logit(y)` where `y = E_roe / E_nu` is the **Bjorken inelasticity** (hadronic energy fraction; lepton fraction = 1 − y).

Predicted energies are derived as `E_roe = y·E_nu`, `E_lep = (1-y)·E_nu`, so `(E_lep + E_roe)/E_nu = 1` exactly by construction. The conservation histogram for predictions should therefore be a delta function at 1 — this cell verifies that, and plots true vs predicted `y = E_roe/E_nu` as a diagnostic.

In [ ]:
import matplotlib.pyplot as plt
import matplotlib.colors as mcolors
import numpy as np
from sklearn.metrics import r2_score
from scipy.stats import pearsonr

E_nu_true  = targets_linear[:, 0]
E_lep_true = targets_linear[:, 1]
E_roe_true = targets_linear[:, 2]
E_nu_pred  = preds_linear[:, 0]
E_lep_pred = preds_linear[:, 1]
E_roe_pred = preds_linear[:, 2]

# y = E_lep/E_nu  (lepton energy fraction = 1 - y_Bjorken)
inel_true = E_roe_true / E_nu_true.clip(min=1e-6)  # Bjorken inelasticity
inel_pred = E_roe_pred / E_nu_pred.clip(min=1e-6)

cons_pred = (E_lep_pred + E_roe_pred) / E_nu_pred.clip(min=1e-6)
assert np.allclose(cons_pred, 1.0, atol=1e-5), \
    f"Energy conservation violated! max deviation = {np.abs(cons_pred - 1).max():.2e}"
print(f"Energy conservation check passed: max deviation = {np.abs(cons_pred - 1).max():.2e}")

cmap = mcolors.LinearSegmentedColormap.from_list(
    'trunc', plt.cm.ocean(np.linspace(0.3, 0.9, 100)))

r2     = r2_score(inel_true, inel_pred)
pear_r, _ = pearsonr(inel_true, inel_pred)
metrics_text = f"$R^2 = {r2:.3f}$\nPearson $R = {pear_r:.3f}$"

pad_x      = 0.05
pred_limit = float(np.percentile(inel_pred, 99))
hi_i       = min(max(np.percentile(inel_true, 99), pred_limit), 3.0)

fig, (ax_global, ax_zoom) = plt.subplots(1, 2, figsize=(14, 6))

ax_global.set_facecolor("#F5F5F5")
hb = ax_global.hexbin(inel_true, inel_pred, gridsize=100, cmap=cmap,
                      bins='log', mincnt=1, alpha=0.8,
                      extent=[0, hi_i, 0, hi_i])
plt.colorbar(hb, ax=ax_global, label='Counts (log)')
ax_global.plot([0, hi_i], [0, hi_i], "k--", linewidth=0.8, label="$y=x$")
ax_global.set_xlim(-pad_x, hi_i + pad_x)
ax_global.set_ylim(-pad_x, hi_i + pad_x)
ax_global.set_title(r"Bjorken inelasticity $y = E_\mathrm{roe} / E_\nu$")
ax_global.set_xlabel(r"True $y$")
ax_global.set_ylabel(r"Predicted $y$")
ax_global.legend(loc='lower right', title=metrics_text, fontsize=8, frameon=False)
ax_global.spines[['top', 'right']].set_visible(False)
ax_global.grid(True, linestyle=':', linewidth=0.8, color='gray', alpha=0.3)

z_start = 0.85
mask    = (inel_true > z_start) | (inel_pred > z_start)
ax_zoom.scatter(inel_true[mask], inel_pred[mask], s=15, alpha=0.6,
                color='#1f77b4', edgecolors='white', linewidth=0.3)
ax_zoom.plot([z_start, hi_i], [z_start, hi_i], "k--", linewidth=0.8)
ax_zoom.set_xlim(z_start, 1.02)
ax_zoom.set_ylim(z_start, hi_i + 0.05)
ax_zoom.set_title(r"Zoomed: high lepton fraction ($y > 0.85$, low $y_\mathrm{Bjorken}$)")
ax_zoom.set_xlabel(r"True $y$")
ax_zoom.set_ylabel(r"Predicted $y$")
ax_zoom.spines[['top', 'right']].set_visible(False)
ax_zoom.grid(True, linestyle=':', linewidth=0.8, color='gray', alpha=0.3)

plt.tight_layout()
plt.savefig(figures_path / "lepton_fraction.png", dpi=500, bbox_inches='tight')

In [ ]:
fig, ax_energy = plt.subplots(figsize=(10, 5))

cons_true = (E_lep_true + E_roe_true) / E_nu_true.clip(min=1e-6)
q01, q99 = np.percentile(cons_true, [1, 99])
buf       = (q99 - q01) * 0.05
bins_c    = np.linspace(q01, q99, 100)
ax_energy.hist(cons_true, bins=bins_c, histtype='stepfilled', alpha=0.6,
               color="#a5deb6", linewidth=0, label="True labels")
ax_energy.axvline(1.0, color='k', linestyle="--", linewidth=0.8,
                  label=f"Predicted (exact, N={len(cons_pred)})")
ax_energy.set_xlim(q01 - buf, q99 + buf)
ax_energy.set_xlabel(r"$(E_\mathrm{lep} + E_\mathrm{roe}) / E_\nu$")
ax_energy.set_ylabel("Events")
ax_energy.set_title("Energy conservation")
ax_energy.legend(fontsize=8, frameon=False)
ax_energy.spines[['top', 'right']].set_visible(False)
ax_energy.grid(True, linestyle=':', linewidth=0.8, color='gray', alpha=0.3)
_ = ax_energy.text(0.05, 0.95,
               f"True  mean = {cons_true.mean():.4f}\nPred  = 1.0000 (exact)",
               transform=ax_energy.transAxes, va="top", fontsize=9,
               bbox=dict(facecolor='white', alpha=0.8, edgecolor='none'))

plt.tight_layout()
plt.savefig(figures_path / "conservation.png", dpi=500, bbox_inches='tight')


**Worst / best event analysis**

For each target, split events into worst and best 25% by absolute relative error, then compare their physical characteristics.

In [ ]:
import pandas as pd

rows = []
for i, data in enumerate(val_dataset):
    y_t = float(inel_true[i])
    y_p = float(inel_pred[i])
    rows.append({
        "n_nodes":    int(data.num_nodes),
        "total_hits": float((10 ** data.x).sum()),
        "faser_0":    float(data.x_faser[0]),
        "faser_1":    float(data.x_faser[1]),
        "faser_2":    float(data.x_faser[2]),
        "E_nu":       float(targets_linear[i, 0]),
        "E_lep":      float(targets_linear[i, 1]),
        "E_roe":      float(targets_linear[i, 2]),
        "y_true":     y_t,
        "log_E_nu":   float(np.log10(max(targets_linear[i, 0], 1e-6))),
        "res_Enu":    float((preds_linear[i, 0] - targets_linear[i, 0]) / targets_linear[i, 0]),
        "res_Elep":   float((preds_linear[i, 1] - targets_linear[i, 1]) / targets_linear[i, 1]),
        "res_Eroe":   float((preds_linear[i, 2] - targets_linear[i, 2]) / targets_linear[i, 2]),
        "res_y":      float((y_p - y_t) / max(y_t, 1e-6)),
        "abserr_Enu": abs(float((preds_linear[i, 0] - targets_linear[i, 0]) / targets_linear[i, 0])),
        "abserr_Elep":abs(float((preds_linear[i, 1] - targets_linear[i, 1]) / targets_linear[i, 1])),
        "abserr_Eroe":abs(float((preds_linear[i, 2] - targets_linear[i, 2]) / targets_linear[i, 2])),
        "abserr_y":   abs(float((y_p - y_t) / max(y_t, 1e-6))),
        "vx_mm":      float(data.true_pos[0].item()) * 100.0 if hasattr(data, "true_pos") else float("nan"),
        "vy_mm":      float(data.true_pos[1].item()) * 100.0 if hasattr(data, "true_pos") else float("nan"),
        "vz_mm":      float(data.true_pos[2].item()) * 100.0 if hasattr(data, "true_pos") else float("nan"),
    })

df = pd.DataFrame(rows)

# Q = 0.25
# for key, col in [("Enu", "abserr_Enu"), ("y", "abserr_y")]:
#     df[f"best_{key}"]  = df[col] <= df[col].quantile(Q)
#     df[f"worst_{key}"] = df[col] >= df[col].quantile(1 - Q)

# df["worst_both"] = df["worst_Enu"] & df["worst_y"]
# df["best_both"]  = df["best_Enu"]  & df["best_y"]

# print(f"Val events: {len(df)}")
# print(f"\n|rel err| thresholds (25th / 75th percentile):")
# print(f"  E_nu : {df.abserr_Enu.quantile(Q):.3f} / {df.abserr_Enu.quantile(1-Q):.3f}")
# print(f"  y    : {df.abserr_y.quantile(Q):.3f}   / {df.abserr_y.quantile(1-Q):.3f}")
# print(f"\nWorst at both E_nu and y : {df.worst_both.sum()} events ({100*df.worst_both.mean():.1f}%)")
# print(f"Best  at both E_nu and y : {df.best_both.sum()}  events ({100*df.best_both.mean():.1f}%)")

In [ ]:
np.save(figures_path.parent / "n_nodes_val.npy", df["n_nodes"].values)
print(figures_path.parent / "n_nodes_val.npy")


In [ ]:
FEATURES = [
    ("E_nu",       r"True $E_\nu$ [TeV]",                   False),
    ("y_true",     r"Bjorken inelasticity $y = E_\mathrm{roe}/E_\nu$",  False),
    ("n_nodes",    "Nodes (super-pixels)",                   False),
    ("total_hits", "Total calorimeter hits",                 True),
    ("faser_0",    "FASER station 0 hits",                   False),
    ("faser_1",    "FASER station 1 hits",                   False),
]

CASES = [
    ("worst_y",   "best_y",   r"inelasticity $y$"),
    ("worst_Enu", "best_Enu", r"$E_\nu$"),
    ("worst_both","best_both", r"$E_\nu$ and $y$ combined"),
]

for worst_col, best_col, err_label in CASES:
    worst = df[df[worst_col]]
    best  = df[df[best_col]]
    if len(worst) == 0 or len(best) == 0:
        continue

    fig, axes = plt.subplots(2, 3, figsize=(14, 7))
    axes = axes.flatten()

    for ax, (feat, label, log_x) in zip(axes, FEATURES):
        w_vals = worst[feat].values
        b_vals = best[feat].values
        lo = min(w_vals.min(), b_vals.min())
        hi = max(np.percentile(w_vals, 99), np.percentile(b_vals, 99))
        bins = np.geomspace(max(lo, 1), hi, 40) if log_x else np.linspace(lo, hi, 40)

        for vals, col, lbl in [
            (b_vals, "#5b8db8", f"Best  (N={len(best)})"),
            (w_vals, "#e05c5c", f"Worst (N={len(worst)})"),
        ]:
            ax.hist(vals, bins=bins, histtype="stepfilled", alpha=0.4, color=col, density=True, label=lbl)
            ax.hist(vals, bins=bins, histtype="step",       linewidth=1.2, color=col, density=True)
            ax.axvline(np.median(vals), color=col, linestyle="--", linewidth=1.0, alpha=0.8)

        if log_x:
            ax.set_xscale("log")
        ax.set_xlabel(label)
        ax.set_ylabel("Density")
        ax.spines[["top", "right"]].set_visible(False)
        ax.grid(True, linestyle=":", linewidth=0.8, color="gray", alpha=0.3)

    axes[0].legend(fontsize=8, frameon=False)
    fig.suptitle(f"Worst vs best events — {err_label} prediction error", fontsize=12)
    plt.tight_layout()
    fname = f"worst_best_{worst_col}.png"
    plt.savefig(figures_path / fname, dpi=350, bbox_inches="tight")
    plt.show()
    print(f"Saved: {fname}")

In [ ]:
df['worst_y_only']   = df['worst_y']   & ~df['worst_Enu']
df['worst_Enu_only'] = df['worst_Enu'] & ~df['worst_y']
df['hits_per_E']     = df['total_hits'] / df['E_nu']

groups = {
    'Best both':           df[df['best_both']],
    'Worst $E_\\nu$ only': df[df['worst_Enu_only']],
    'Worst $y$ only':      df[df['worst_y_only']],
    'Worst both':          df[df['worst_both']],
}

cols   = ['E_nu', 'y_true', 'n_nodes', 'total_hits', 'hits_per_E', 'faser_0', 'faser_1', 'faser_2']
labels = [
    r'$E_\nu$ [TeV]',
    r'$y = E_\mathrm{roe}/E_\nu$ (Bjorken inelasticity)',
    'N nodes',
    'Total hits',
    r'Hits/$E_\nu$',
    'FASER station 0',
    'FASER station 1',
    'FASER station 2',
]
colors = ['#5b8db8', '#8dbd8d', '#e8a838', '#e05c5c']

fig, axes = plt.subplots(2, 4, figsize=(16, 8))
axes = axes.flatten()

for ax, col, label in zip(axes, cols, labels):
    means   = [g[col].mean() for g in groups.values()]
    stderrs = [g[col].sem()  for g in groups.values()]
    x = np.arange(len(groups))
    ax.bar(x, means, yerr=stderrs, color=colors, alpha=0.8,
           capsize=4, error_kw=dict(linewidth=1.0))
    ax.set_xticks(x)
    ax.set_xticklabels(list(groups.keys()), rotation=25, ha='right', fontsize=7)
    ax.set_ylabel(label)
    ax.spines[['top', 'right']].set_visible(False)
    ax.grid(True, axis='y', linestyle=':', alpha=0.3)

for g_name, g_df in groups.items():
    print(f"{g_name:<22} N={len(g_df)}")

fig.suptitle('Event characteristics by failure mode  (mean ± SEM)\n'
             r'$y = E_\mathrm{roe}/E_\nu$ (Bjorken inelasticity = $1 -$ lepton fraction)', fontsize=11)
plt.tight_layout()
plt.savefig(figures_path / "failure_modes.png", dpi=350, bbox_inches='tight')
plt.show()
print(f"Saved: {figures_path / 'failure_modes.png'}")

In [ ]:
# ── Save comprehensive stats to report_stats.json ────────────────────────────
# Appends/updates the entry for WEIGHTS_DIR in get_weights_path()/"report_stats.json".
# All downstream tools (generate_latex_macros.py) read from this file.
# Safe to re-run: overwrites the entry for the current WEIGHTS_DIR only.

import json as _json
import datetime as _dt
import numpy as _np
from scipy.optimize import curve_fit as _curve_fit
from scipy.stats import pearsonr as _pearsonr

def _gauss(x, A, mu, sigma):
    return A * _np.exp(-0.5 * ((x - mu) / sigma) ** 2)

def _per_bin_energy(yt, yp, bins):
    rows = []
    res = (yp - yt) / yt
    for emin, emax in bins:
        mask = (yt >= emin) & (yt < emax)
        n = int(mask.sum())
        if n == 0:
            continue
        r = res[mask]
        s = float(_np.std(r))
        row = {
            "emin": emin, "emax": emax, "n": n,
            "sigma": s,
            "sigma_se": float(s / _np.sqrt(2 * n)),
            "bias": float(_np.mean(r)),
            "bias_se": float(s / _np.sqrt(n)),
            "median_abs_rel_err": float(_np.median(_np.abs(r))),
            "gauss_mu": None, "gauss_sigma": None, "gauss_sigma_se": None,
        }
        try:
            p1, p99 = _np.percentile(r, [1, 99])
            xlim = min(max(abs(p1), abs(p99), 0.1), 5.0)
            counts, edges = _np.histogram(r, bins=_np.linspace(-xlim, xlim, 60))
            centers = 0.5 * (edges[:-1] + edges[1:])
            popt, _ = _curve_fit(_gauss, centers, counts,
                                 p0=[counts.max(), _np.mean(r), s], maxfev=5000)
            row["gauss_mu"]      = float(popt[1])
            row["gauss_sigma"]   = float(abs(popt[2]))
            row["gauss_sigma_se"]= float(abs(popt[2]) / _np.sqrt(2 * n))
        except Exception:
            pass
        rows.append(row)
    return rows

def _per_bin_y(yt_col, yp_col, y_true_vals, n_bins=10):
    rows = []
    y_bins = _np.linspace(0, 1, n_bins + 1)
    for ymin, ymax in zip(y_bins[:-1], y_bins[1:]):
        mask = (y_true_vals >= ymin) & (y_true_vals < ymax)
        n = int(mask.sum())
        if n < 20:
            continue
        r = (yp_col[mask] - yt_col[mask]) / yt_col[mask]
        s = float(_np.std(r))
        rows.append({
            "ymin": round(float(ymin), 2), "ymax": round(float(ymax), 2), "n": n,
            "sigma": s,
            "sigma_se": float(s / _np.sqrt(2 * n)),
            "bias": float(_np.mean(r)),
            "bias_se": float(s / _np.sqrt(n)),
        })
    return rows

# ── Recompute everything from preds_linear / targets_linear ──────────────────
_yt  = targets_linear          # [N, 3]
_yp  = preds_linear            # [N, 3]
_N   = len(_yt)

# Inelasticity
_inel_true = _yt[:, 2] / _yt[:, 0].clip(min=1e-6)
_inel_pred = _yp[:, 2] / _yp[:, 0].clip(min=1e-6)

# Baseline (total hits linear fit) - recompute in case baseline cell not run
_hits = _np.array([float((10 ** data.x[:, 0]).sum()) for data in val_dataset])

_stats = {
    "meta": {
        "weights_dir":    WEIGHTS_DIR,
        "epoch":          int(checkpoint["epoch"]) + 1,
        "val_loss":       float(checkpoint["val_loss"]),
        "val_rmse":       float(checkpoint["val_rmse"]),
        "loss_fn":        checkpoint.get("loss_fn"),
        "huber_delta":    checkpoint.get("huber_delta"),
        "parametrisation": checkpoint.get("parametrisation"),
        "n_val_events":   _N,
        "timestamp":      _dt.datetime.now().isoformat(),
    },
    "targets":       {},
    "inelasticity_y": {},
    "baseline":      {},
}

for _i, _name in enumerate(TARGET_NAMES):
    _yt_i = _yt[:, _i]
    _yp_i = _yp[:, _i]
    _res  = (_yp_i - _yt_i) / _yt_i

    _s,  _sl, _sh = bootstrap_metric(_yt_i, _yp_i, resolution_fn)
    _r,  _rl, _rh = bootstrap_metric(_yt_i, _yp_i, r2_fn)
    _b,  _bl, _bh = bootstrap_metric(_yt_i, _yp_i, bias_fn)
    _pr, _    = _pearsonr(_yt_i, _yp_i)

    # Δσ between this run and linear baseline (paired bootstrap)
    _c    = _np.polyfit(_hits, _yt_i, 1)
    _lin  = _np.polyval(_c, _hits)
    _rng  = _np.random.default_rng(42)
    _dsig = []
    for _ in range(1000):
        _idx = _rng.integers(0, _N, _N)
        _sg  = _np.std((_yp_i[_idx] - _yt_i[_idx]) / _yt_i[_idx])
        _sl2 = _np.std((_lin[_idx]  - _yt_i[_idx]) / _yt_i[_idx])
        _dsig.append(_sl2 - _sg)   # positive = GravNet better
    _dsig = _np.array(_dsig)
    _lin_s = float(_np.std((_lin - _yt_i) / _yt_i))
    _gn_s  = float(_np.std((_yp_i - _yt_i) / _yt_i))

    _stats["targets"][_name] = {
        # Headline metrics with 95% bootstrap CI
        "sigma":              _s,  "sigma_ci_lo":  _sl, "sigma_ci_hi":  _sh,
        "r2":                 _r,  "r2_ci_lo":     _rl, "r2_ci_hi":     _rh,
        "bias":               _b,  "bias_ci_lo":   _bl, "bias_ci_hi":   _bh,
        "pearson_r":          float(_pr),
        "median_abs_rel_err": float(_np.median(_np.abs(_res))),
        "mean_abs_rel_err":   float(_np.mean(_np.abs(_res))),
        # Per energy bin
        "per_bin_energy":     _per_bin_energy(_yt_i, _yp_i, ENERGY_BINS_TEV),
        # Per inelasticity-y bin
        "per_bin_y":          _per_bin_y(_yt_i, _yp_i, _inel_true),
    }
    _stats["baseline"][_name] = {
        "gravnet_sigma":       _gn_s,
        "linear_sigma":        _lin_s,
        "improvement_pct":     float((_lin_s - _gn_s) / _lin_s * 100),
        "gravnet_r2":          float(1 - _np.sum((_yp_i - _yt_i)**2) / _np.sum((_yt_i - _yt_i.mean())**2)),
        "linear_r2":           float(1 - _np.sum((_lin  - _yt_i)**2) / _np.sum((_yt_i - _yt_i.mean())**2)),
        "delta_sigma_median":  float(_np.median(_dsig)),
        "delta_sigma_ci_lo":   float(_np.percentile(_dsig, 2.5)),
        "delta_sigma_ci_hi":   float(_np.percentile(_dsig, 97.5)),
    }

# Inelasticity y stats
_s_y, _sl_y, _sh_y = bootstrap_metric(_inel_true, _inel_pred, resolution_fn)
_r_y, _rl_y, _rh_y = bootstrap_metric(_inel_true, _inel_pred, r2_fn)
_b_y, _bl_y, _bh_y = bootstrap_metric(_inel_true, _inel_pred, bias_fn)
_pr_y, _ = _pearsonr(_inel_true, _inel_pred)
_stats["inelasticity_y"] = {
    "sigma":              _s_y,  "sigma_ci_lo":  _sl_y, "sigma_ci_hi":  _sh_y,
    "r2":                 _r_y,  "r2_ci_lo":     _rl_y, "r2_ci_hi":     _rh_y,
    "bias":               _b_y,  "bias_ci_lo":   _bl_y, "bias_ci_hi":   _bh_y,
    "pearson_r":          float(_pr_y),
    "median_abs_rel_err": float(_np.median(_np.abs((_inel_pred - _inel_true) / _inel_true.clip(min=1e-6)))),
    "per_bin_energy":     _per_bin_energy(_inel_true, _inel_pred,
                                          [(b[0], b[1]) for b in [(0.0,0.1),(0.1,0.2),(0.2,0.3),
                                           (0.3,0.4),(0.4,0.5),(0.5,0.6),(0.6,0.7),(0.7,0.8),(0.8,0.9),(0.9,1.0)]]),
}

# ── Write to report_stats.json ────────────────────────────────────────────────
_stats_file = get_weights_path() / "report_stats.json"
_existing   = _json.loads(_stats_file.read_text()) if _stats_file.exists() else {}
_existing[WEIGHTS_DIR] = _stats
_stats_file.write_text(_json.dumps(_existing, indent=2))
print(f"Saved stats → {_stats_file}")
print(f"  Keys in file: {list(_existing.keys())}")
print(f"  Targets: {list(_stats['targets'].keys())}")
print(f"  σ(E_roe) = {_stats['targets']['E_roe']['sigma']:.4f}  "
      f"[{_stats['targets']['E_roe']['sigma_ci_lo']:.4f}, {_stats['targets']['E_roe']['sigma_ci_hi']:.4f}]")

Poss future additions

- Best/worst event display: sort val_dataset by |pred − true| / true, visualise graph hits of best and worst events.
- Containment study: filter by vertex vz within detector acceptance. Check data.keys() and available parquet columns first.